# Lecture 4: Regular Expressions & Structured Text — Answer Key
### NLP Course 2027 — Week 02

---

This notebook provides complete, worked answers for the four exercises in **Lecture 4: Regular Expressions & Structured Text**.

In [ ]:
import re
import nltk

for pkg in ['averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk import pos_tag, RegexpParser

print('Setup complete.')

---
## Exercise 4.1

**Task:** Write a regex to extract all phone numbers. Handle three formats:
- `(123) 456-7890`
- `123-456-7890`
- `+1 123 456 7890`

**Key concept:** Phone number formats vary by country and style convention. A robust extractor uses *alternation* (`|`) or *optional groups* (`(?:...)?`) to cover multiple formats in a single pattern. The key building blocks are: optional country code, area code (with or without parentheses), and digit groups separated by a flexible separator.

In [ ]:
import re

# Pattern breakdown:
#   (?:\+1[\s-])?         -- optional country code +1 followed by space or dash
#   (?:\(?\d{3}\)?[\s.-]) -- area code: optional parens around 3 digits, then separator
#   \d{3}[\s.-]           -- exchange: 3 digits then separator
#   \d{4}                 -- subscriber: 4 digits
#
# Using a single unified pattern rather than alternation keeps the match
# groups consistent and the pattern easy to extend.

PHONE_PATTERN = r'(?:\+1[\s-])?(?:\(?\d{3}\)?[\s.-])\d{3}[\s.-]\d{4}'

def extract_phone_numbers(text: str) -> list:
    """
    Extract all phone numbers from text, handling common US formats.

    Supported:
      (123) 456-7890
      123-456-7890
      +1 123 456 7890
      +1-800-555-0100
    """
    # re.findall returns all non-overlapping matches as strings
    matches = re.findall(PHONE_PATTERN, text)
    # Strip leading/trailing whitespace from each match
    return [m.strip() for m in matches]


# ── Test corpus ──────────────────────────────────────────────────────────────
test_cases = [
    'Call us at (123) 456-7890 for support.',
    'Emergency: 123-456-7890 or 987-654-3210.',
    'International: +1 123 456 7890 and +1-800-555-0100.',
    'No phone here: just text 12345 and 9999.',
    'Mixed: (415) 555-2671, 650-867-5309, +1 212 555-1234.',
]

for text in test_cases:
    numbers = extract_phone_numbers(text)
    print(f'Input : {text}')
    print(f'Found : {numbers}\n')

**Pattern anatomy:**

```
(?:\+1[\s-])?          optional +1 country code with separator
(?:\(?\d{3}\)?[\s.-])  area code: (123) or 123, followed by space/dot/dash
\d{3}[\s.-]            exchange number + separator
\d{4}                  subscriber number
```

`(?:...)` is a *non-capturing group* — it groups for quantification without storing the match in a capture group, keeping `re.findall` returning the full match string rather than a tuple of group contents.

---
## Exercise 4.2

**Task:** Write `count_pattern_matches(text, pattern)` that counts **overlapping** regex matches using `re.finditer`.

**Key concept:** By default, `re.findall` returns non-overlapping matches. For overlapping matches you need `re.finditer` combined with a *lookahead assertion* (`(?=...)`) which matches a position without consuming characters, allowing the engine to re-try at the very next character.

In [ ]:
import re


def count_pattern_matches(text: str, pattern: str) -> dict:
    """
    Count both non-overlapping and overlapping occurrences of pattern in text.

    Non-overlapping: use re.findall (standard).
    Overlapping    : wrap pattern in a lookahead (?=...) so the engine does
                     not advance past the match, then count positions found.

    Returns
    -------
    dict with:
        non_overlapping : int   – standard re.findall count
        overlapping     : int   – count using lookahead trick
        overlap_matches : list  – the overlapping match strings
    """
    # Standard non-overlapping
    non_overlap = re.findall(pattern, text)

    # Overlapping: use lookahead (?=pattern) — each zero-width match records
    # the start position; extract the actual text using match.start() + span.
    overlap_positions = [m.start() for m in re.finditer(f'(?={pattern})', text)]
    # Re-match each position to get the actual matched string
    overlap_matches = []
    for pos in overlap_positions:
        m = re.match(pattern, text[pos:])
        if m:
            overlap_matches.append(m.group())

    return {
        'non_overlapping': len(non_overlap),
        'overlapping': len(overlap_matches),
        'overlap_matches': overlap_matches,
    }


# ── Examples ─────────────────────────────────────────────────────────────────
examples = [
    ('aaaa', 'aa'),               # 'aa' fits at pos 0,1,2 → 3 overlapping, 2 non
    ('abcabcabc', 'abc'),         # non-overlapping: 3; overlapping also 3
    ('aababab', 'aba'),           # overlapping pattern
    ('the cat sat on the mat', r'\b\w+at\b'),  # words ending in 'at'
]

for text, pattern in examples:
    result = count_pattern_matches(text, pattern)
    print(f'Text    : {repr(text)}')
    print(f'Pattern : {repr(pattern)}')
    print(f'Non-overlapping : {result["non_overlapping"]}')
    print(f'Overlapping     : {result["overlapping"]}  {result["overlap_matches"]}')
    print()

**Why lookahead works for overlapping:**

```
text    :  a  a  a  a
pattern :  a  a
positions: 0  1  2  3

(?=aa) fires at positions 0, 1, 2  → 3 overlapping matches
re.findall('aa') matches at 0, 2   → 2 non-overlapping matches
```

The lookahead `(?=pattern)` is a *zero-width assertion* — it asserts that `pattern` matches at the current position without consuming any characters, so the engine can test every character position.

---
## Exercise 4.3

**Task:** Build a regex-based email classifier: extract *action items* — sentences containing `'please'`, `'need to'`, `'must'`, or `'should'`.

**Key concept:** This is a primitive form of *intent classification* or *imperative sentence detection*. Regex-based classifiers are fast, interpretable, and require no training data — at the cost of recall (they miss paraphrases). The key challenge is sentence segmentation before applying the keyword pattern.

In [ ]:
import re
import nltk


def classify_email_sentences(email_text: str) -> dict:
    """
    Parse an email and extract action-item sentences.

    A sentence is an action item if it contains any of:
        please | need to | must | should

    Returns
    -------
    dict with:
        all_sentences   : list of all sentences found
        action_items    : list of action-item sentences
        action_count    : int
    """
    # Use NLTK sent_tokenize for robust sentence segmentation
    sentences = nltk.sent_tokenize(email_text)

    # Pattern: word boundary around each keyword ensures 'must' doesn't
    # match 'mustard', 'please' doesn't match 'displeased', etc.
    action_pattern = re.compile(
        r'\b(?:please|need\s+to|must|should)\b',
        re.IGNORECASE
    )

    action_items = [s for s in sentences if action_pattern.search(s)]

    return {
        'all_sentences': sentences,
        'action_items': action_items,
        'action_count': len(action_items),
    }


# ── Sample email ─────────────────────────────────────────────────────────────
email = """
Hi team,

Thanks for the great work on the Q3 report. The results look promising.
Please review the attached document before our meeting on Friday.
John needs to update the budget figures in section 3.
Everyone should read the new compliance guidelines by end of week.
I also wanted to mention that our office party is scheduled for December 15.
You must submit your expense reports by December 1st or they will not be reimbursed.
The cafeteria menu has changed — there are now more vegetarian options.
Please let me know if you have any questions.

Best,
Manager
"""

result = classify_email_sentences(email)

print(f'Total sentences  : {len(result["all_sentences"])}')
print(f'Action items     : {result["action_count"]}\n')
print('ACTION ITEMS:')
print('-' * 60)
for i, sent in enumerate(result['action_items'], 1):
    print(f'{i}. {sent}')

print()
print('NON-ACTION sentences:')
for sent in result['all_sentences']:
    if sent not in result['action_items'] and len(sent.split()) > 3:
        print(f'   {sent}')

**Design notes:**

- `\b` word boundaries prevent false positives like `'displeased'` matching `'please'`.
- `re.IGNORECASE` makes the pattern case-insensitive — `MUST`, `Should`, and `must` all match.
- `\s+` between `'need'` and `'to'` handles both `need to` and `need  to` (multiple spaces).
- Using `nltk.sent_tokenize` before applying the pattern avoids the classic bug of a keyword spanning a sentence boundary being caught in the wrong sentence.

**Limitations of regex-based action item detection:**

- Misses paraphrases: `'It would be great if you could ...'` is an implicit request but contains none of the keywords.
- False positives: `'You should know that the project succeeded'` is informational, not imperative, but matches `should`.
- A trained classifier (e.g., fine-tuned BERT) would handle these cases better.

---
## Exercise 4.4

**Task:** Using `RegexpParser`, write a grammar to extract VP chunks from 3 sentences.

**Key concept:** VP chunking uses POS tag patterns rather than word patterns. A basic VP is a verb (any `VB*` tag) optionally followed by adverbs (`RB*`) and modal auxiliaries are often tagged `MD`. The grammar uses `{...}` to define what is *inside* a chunk and `}...{` (chinking) to define what should be *excluded*.

In [ ]:
import nltk
from nltk import RegexpParser, pos_tag, word_tokenize


# Grammar explanation:
#   VP: {<MD>?<VB.*><RB.*>*}   — optional modal, then a verb form, then 0+ adverbs
#   This captures:
#     'runs quickly'          VBZ RB
#     'should have been'      MD VB VBN  (won't match without extension)
#     'can quickly process'   MD RB VB
#
# A richer grammar could chain auxiliaries with {<MD><VB.*>+<RB.*>*} but
# we keep it readable for this exercise.

VP_GRAMMAR = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}            # Noun phrase (needed for VP context)
  VP: {<MD>?<VB.*><RB.*>*}             # Verb phrase: optional modal + verb(s) + adverbs
"""

chunker = RegexpParser(VP_GRAMMAR)


def extract_vp_chunks(sentence: str) -> list:
    """
    POS-tag sentence and extract VP chunks using the grammar above.

    Returns list of (phrase_text, leaf_tags) tuples.
    """
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    tree = chunker.parse(tagged)

    vps = []
    for subtree in tree.subtrees():
        if subtree.label() == 'VP':
            phrase = ' '.join(w for w, t in subtree.leaves())
            tag_seq = ' '.join(t for w, t in subtree.leaves())
            vps.append({'phrase': phrase, 'tags': tag_seq})
    return vps


# ── Three test sentences with different VP structures ────────────────────────
sentences = [
    'The researchers quickly published their findings.',
    'She should carefully review the contract before signing.',
    'Deep learning models can accurately predict complex patterns.',
]

for sent in sentences:
    vps = extract_vp_chunks(sent)
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)

    print(f'Sentence : {sent}')
    print(f'POS tags : {[(w, t) for w, t in tagged if t.startswith(("V", "M", "R"))]}')
    if vps:
        for vp in vps:
            print(f'  VP chunk: "{vp["phrase"]}" [{vp["tags"]}]')
    else:
        print('  No VP chunks found (check POS tags above)')
    print()

In [ ]:
# Visualise the parse tree for one sentence
sent = 'The researchers quickly published their findings.'
tokens = word_tokenize(sent)
tagged = pos_tag(tokens)
tree = chunker.parse(tagged)

print('Full parse tree for:')
print(f'  "{sent}"\n')
print(tree)

# Pretty-print each subtree with its label
print()
print('Chunk subtrees:')
for subtree in tree.subtrees():
    if subtree.label() in ('NP', 'VP'):
        print(f'  [{subtree.label()}] {subtree}')

**Grammar design notes:**

- `<VB.*>` uses a *wildcard* inside the tag pattern — it matches `VB`, `VBD`, `VBG`, `VBN`, `VBP`, `VBZ` (all verb forms).
- `<RB.*>*` captures zero or more adverbs (`RB`, `RBR`, `RBS`).
- `<MD>?` makes the modal auxiliary optional, so both `'published'` and `'should publish'` are matched.
- The grammar is *context-free only at the chunk level* — it cannot capture long-distance dependencies (e.g. `'would, if possible, prefer'`).

**Chunking vs full parsing:**

| | Chunking | Full parsing |
|---|---|---|
| Scope | Local phrase patterns | Full sentence structure |
| Speed | Very fast | Slower |
| Errors | Low false-positive rate | Cascading errors possible |
| Use case | IE, feature extraction | Grammar checking, semantic parsing |

---
## Summary of Key Concepts

| Exercise | Concept | Key Takeaway |
|----------|---------|-------------|
| 4.1 | Phone number extraction | Use optional groups `(?:...)?` and flexible separators `[\s.-]` |
| 4.2 | Overlapping matches | Lookahead `(?=pattern)` is zero-width — enables re-trying at every position |
| 4.3 | Regex-based classification | Fast and interpretable; misses paraphrases; word boundaries prevent false positives |
| 4.4 | VP chunking | POS-tag patterns with `<VB.*>` wildcards; `<MD>?` optional modal; part of shallow parsing |

---
*NLP Course 2027 — Week 02*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**